<a href="https://colab.research.google.com/github/adhishmodi/customer-churn-prediction/blob/feature/milestone-5-validation-framework/notebooks/05_model_improvement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Churn Prediction — Milestone 5: Validation Stability

This notebook begins Milestone 5 with **Phase 1: repeated cross-validation**. It evaluates fixed initial model configurations on the training partition only. It does not tune hyperparameters, select a final improved model, or inspect held-out test labels.

## Research question

How much do the fixed model configurations vary across repeated stratified training/validation splits, and how consistently does each model rank first by ROC-AUC?

## Important baseline distinction

The historical Milestone 3–4 selected model remains **Tuned XGBoost** with CV ROC-AUC 0.8501 and held-out test ROC-AUC 0.8463. The exact original model artifact is not stored in Git. The get_models() candidates below are the fixed initial configurations, including the initial XGBoost configuration; they are not a substitute for the tuned baseline artifact. Do not compare their repeated-CV means directly with the old single 5-fold tuned score as if the protocols were identical.

## Leakage safeguards

- Recreate the original stratified 80/20 split with seed 42.
- Pass only X_train and y_train to repeated CV.
- Keep preprocessing inside each model pipeline.
- Do not use the held-out test split for model selection or threshold selection.
- Repeated-fold standard deviations are descriptive, not formal confidence intervals; folds overlap across repeats.


## 1. Runtime setup and dataset loading

Run this notebook from a clean Colab runtime. Upload the original IBM Telco CSV if it is not already at the expected path.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/adhishmodi/customer-churn-prediction.git"
BRANCH = "feature/milestone-5-validation-framework"
EXPECTED_CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "preprocess.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path("/content/customer-churn-prediction")
    if not (PROJECT_ROOT / "src" / "preprocess.py").is_file():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH,
             REPOSITORY_URL, str(PROJECT_ROOT)],
            check=True,
        )
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_PATH, REPORTS_DIR, RANDOM_SEED
from src.preprocess import clean_data, prepare_features_target, split_data
from src.train import get_models
from src.experiments import (
    compare_models_repeated_cv,
    evaluate_nested_cv_model_selection,
)

RAW_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
if not RAW_DATA_PATH.is_file():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            f"Place the original dataset at {RAW_DATA_PATH} before running."
        ) from exc
    uploaded = files.upload()
    if EXPECTED_CSV not in uploaded:
        raise FileNotFoundError(
            f"Upload the exact source file named {EXPECTED_CSV}."
        )
    RAW_DATA_PATH.write_bytes(uploaded[EXPECTED_CSV])

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {RAW_DATA_PATH}")
print(f"Random seed: {RANDOM_SEED}")

## 2. Reconstruct the original training partition

The split is reproduced with the same deterministic cleaning and split function used in earlier milestones. The test partition is created for reproducibility checks but is not passed to any experiment function.

In [ ]:
import pandas as pd
from IPython.display import display

raw_df = pd.read_csv(RAW_DATA_PATH)
cleaned_df = clean_data(raw_df)
X, y = prepare_features_target(cleaned_df)
X_train, X_test, y_train, y_test = split_data(X, y)

assert len(X_train) == 5634
assert len(X_test) == 1409
assert X_train.index.intersection(X_test.index).empty
assert y_train.index.equals(X_train.index)
assert y_test.index.equals(X_test.index)
assert "customerID" not in X_train.columns
assert "Churn" not in X_train.columns

print(f"Training rows: {len(X_train):,}")
print(f"Held-out test rows reserved: {len(X_test):,}")
print(f"Training churn prevalence: {y_train.mean():.2%}")
print("The held-out test partition will not be passed to repeated CV.")

## 3. Repeated stratified cross-validation

Repeated Stratified K-Fold repeats the stratified fold assignment several times with different shuffles. Each model is refitted from scratch in every fold. Because preprocessing is inside the pipeline, imputers, scaling and encoding are learned only from that fold's training portion.

We use 5 folds × 3 repeats (15 validation folds per model). This provides a descriptive view of variability and ranking stability for the fixed initial configurations. It does not replace nested CV for estimating the performance of a full hyperparameter/model-selection procedure.

In [ ]:
repeated_results = compare_models_repeated_cv(
    X_train,
    y_train,
    models=get_models(),
    n_splits=5,
    n_repeats=3,
    random_state=RANDOM_SEED,
)

fold_scores = repeated_results["fold_scores"]
repeated_summary = repeated_results["summary"].sort_values(
    "roc_auc_mean", ascending=False
)
repeat_winners = repeated_results["repeat_winners"]
selection_stability = repeated_results["selection_stability"]

display(repeated_summary.round(4))
display(repeat_winners.round(4))
display(selection_stability.round(4))
print("Experiment configuration:", repeated_results["configuration"])

## 4. Nested cross-validation: evaluate the selection procedure

Repeated CV above describes the variability of fixed model configurations. Nested CV answers a different question: how well does the complete process of tuning several model families and choosing the winner generalize? The inner folds tune each model and choose the winner; the outer fold is kept separate and evaluates that winner.

This notebook uses 3 outer folds, 3 inner folds and 5 randomized configurations per candidate model. With three tunable model families, this requires 135 inner-CV candidate fits plus refits. It may take several minutes in Colab. The exact original tuned baseline artifact is unavailable in Git, so this nested comparison evaluates the current Logistic Regression, Random Forest and XGBoost search spaces; it does not recreate the historical tuned artifact.

In [ ]:
nested_results = evaluate_nested_cv_model_selection(
    X_train,
    y_train,
    model_names=["Logistic Regression", "Random Forest", "XGBoost"],
    outer_splits=3,
    inner_splits=3,
    n_iter=5,
    random_state=RANDOM_SEED,
)

nested_candidates = nested_results["inner_candidate_results"]
nested_outer = nested_results["outer_fold_results"]
nested_summary = nested_results["outer_summary"]
nested_selection = nested_results["selection_frequency"]

display(nested_candidates.round(4))
display(nested_outer.round(4))
display(nested_summary.round(4))
display(nested_selection.round(4))
print("Nested CV configuration:", nested_results["configuration"])

## 5. Visualize model variability

The error bars show ±1 standard deviation across the 15 fold scores. These are descriptive spread indicators, not confidence intervals. A small difference in means should not be treated as a statistically significant improvement.

In [ ]:
import matplotlib.pyplot as plt

plot_data = repeated_summary.sort_values("roc_auc_mean")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(
    plot_data["model"],
    plot_data["roc_auc_mean"],
    xerr=plot_data["roc_auc_std"],
    capsize=4,
)
ax.set_xlabel("ROC-AUC (mean ± 1 standard deviation across folds)")
ax.set_title("Repeated Stratified CV — Fixed Initial Model Configurations")
ax.set_xlim(0.0, 1.0)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 6. Save experiment outputs

The generated reports are local artifacts and are ignored by Git. Preserve them with the Colab project files if you want to compare them with later Milestone 5 experiments.

In [ ]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
fold_path = REPORTS_DIR / "m5_repeated_cv_fold_scores.csv"
summary_path = REPORTS_DIR / "m5_repeated_cv_summary.csv"
winners_path = REPORTS_DIR / "m5_repeated_cv_repeat_winners.csv"
stability_path = REPORTS_DIR / "m5_repeated_cv_selection_stability.csv"
nested_outer_path = REPORTS_DIR / "m5_nested_cv_outer_results.csv"
nested_candidates_path = REPORTS_DIR / "m5_nested_cv_inner_candidates.csv"
nested_summary_path = REPORTS_DIR / "m5_nested_cv_summary.csv"
nested_selection_path = REPORTS_DIR / "m5_nested_cv_selection_frequency.csv"

fold_scores.to_csv(fold_path, index=False)
repeated_summary.to_csv(summary_path, index=False)
repeat_winners.to_csv(winners_path, index=False)
selection_stability.to_csv(stability_path, index=False)
nested_outer.to_csv(nested_outer_path, index=False)
nested_candidates.to_csv(nested_candidates_path, index=False)
nested_summary.to_csv(nested_summary_path, index=False)
nested_selection.to_csv(nested_selection_path, index=False)

for path in (fold_path, summary_path, winners_path, stability_path, nested_outer_path, nested_candidates_path, nested_summary_path, nested_selection_path):
    assert path.is_file()
    print(f"Saved: {path}")

## 7. Interpretation checklist

Before proceeding to class-imbalance or feature-engineering experiments, answer:

1. Which fixed model has the highest mean repeated-CV ROC-AUC?
2. Are the mean ROC-AUC differences large relative to the fold-to-fold standard deviations?
3. Does the same model win in all three repeats, or does the ranking change?
4. Does Average Precision tell a similar story to ROC-AUC?
5. Are there practical precision/recall trade-offs worth investigating?

Do not call the highest mean score a proven improvement. These results describe fixed initial configurations under repeated CV; they do not measure the generalization performance of a hyperparameter-selection procedure. The original tuned XGBoost baseline remains unchanged.
